# Imports

In [1]:
import optuna
import pickle

import numpy as np
import pandas as pd

from tqdm import tqdm

from sklearn.metrics import balanced_accuracy_score
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_predict

/home/junior/Documentos/GitHub/kaggle-competition-predicting-student-health/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Utils

In [2]:
def load_pickle(file_path):
    with open(file_path, 'rb') as file:
        return pickle.load(file)

# Loading Datasets

In [3]:
X_train = pd.read_parquet('../data/processed/X_train_stacking_layer_one.parquet')
y_train = pd.read_parquet('../data/interim/y_train.parquet')

X_test = pd.read_parquet('../data/processed/X_test_stacking_layer_one.parquet')

In [4]:
X_train.head()

,lgbm_0,lgbm_1,lgbm_2,xgb_0,xgb_1,xgb_2,cat_0,cat_1,cat_2
0,0.004831,0.002142,0.993027,0.005220,0.002128,0.992652,0.005655,0.000463,0.993882
1,0.998229,0.000143,0.001628,0.997774,0.000322,0.001904,0.994638,0.000368,0.004994
2,0.003982,0.000584,0.995434,0.002362,0.000710,0.996928,0.002327,0.000919,0.996754
3,0.007424,0.001954,0.990622,0.002922,0.003628,0.993450,0.003433,0.002374,0.994194
4,0.993583,0.000012,0.006405,0.997151,0.000013,0.002836,0.994112,0.000005,0.005883


In [5]:
X_test.head()

,lgbm_0,lgbm_1,lgbm_2,xgb_0,xgb_1,xgb_2,cat_0,cat_1,cat_2
0,0.011986,0.002516,0.985498,0.009749,0.003956,0.986295,0.008185,0.005526,0.986288
1,0.469590,0.002922,0.527488,0.344531,0.002104,0.653365,0.530403,0.002094,0.467503
2,0.998492,0.000991,0.000517,0.998555,0.001132,0.000313,0.996306,0.003216,0.000478
3,0.986619,0.000009,0.013372,0.978274,0.000024,0.021702,0.990683,0.000093,0.009224
4,0.008520,0.001975,0.989505,0.005601,0.002080,0.992319,0.003507,0.001947,0.994546


# Machine Learning

In [6]:
label_encoder = load_pickle('../artifacts/label_encoder.pkl')

In [7]:
def objective(trial, X, y):
    
    l1_ratio = trial.suggest_float("l1_ratio", 0.0, 1.0)
    C = trial.suggest_float("C", 1e-5, 100.0, log=True)
    class_weight = trial.suggest_categorical("class_weight", [None, "balanced"])
    fit_intercept = trial.suggest_categorical("fit_intercept", [True, False])
    tol = trial.suggest_float("tol", 1e-6, 1e-2, log=True)
    max_iter = trial.suggest_int("max_iter", 1000, 5000)

    w0 = trial.suggest_float('weight_class_0', 0.05, 10.0, log=True)
    w1 = trial.suggest_float('weight_class_1', 0.05, 10.0, log=True)
    w2 = trial.suggest_float('weight_class_2', 0.05, 10.0, log=True)
    weights = np.array([w0, w1, w2])

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    scores = []

    for fold, (train_idx, valid_idx) in enumerate(cv.split(X, y)):
        
        X_train_fold, X_valid_fold = X.iloc[train_idx, :], X.iloc[valid_idx, :]
        y_train_fold, y_valid_fold = y.iloc[train_idx], y.iloc[valid_idx]

        model = LogisticRegression(
            solver="saga",
            C=C,
            l1_ratio=l1_ratio,
            class_weight=class_weight,
            fit_intercept=fit_intercept,
            tol=tol,
            max_iter=max_iter,
            random_state=42,
        ).fit(X_train_fold, y_train_fold)

        proba = model.predict_proba(X_valid_fold)
        
        weighted_probas = proba * weights
        pred = np.argmax(weighted_probas, axis=1)
        
        score = balanced_accuracy_score(y_valid_fold, pred)
        scores.append(score)

        trial.report(np.mean(scores), step=fold)
        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()

    return np.mean(scores)


study = optuna.create_study(
    direction="maximize", 
    sampler=optuna.samplers.TPESampler(seed=42), 
    pruner=optuna.pruners.MedianPruner(n_warmup_steps=2)
)

study.optimize(
    lambda trial: objective(trial, X_train.loc[:, ['xgb_0', 'xgb_1', 'xgb_2']], y_train.health_condition), 
    n_trials=30, 
    n_jobs=-1, 
    show_progress_bar=True
)

print("\nBest trial score:")
print(study.best_trial.value)

print("\nBest params:")
print(study.best_trial.params)

[I 2026-07-03 10:34:03,453] A new study created in memory with name: no-name-853950ba-a249-476c-b37c-940382560533
  0%|                                                                                                                                                                                       | 0/30 [00:04<?, ?it/s]

KeyboardInterrupt



In [7]:
best_params = {'l1_ratio': 0.8474502151007101, 'C': 0.0009803059401675468, 'class_weight': 'balanced', 'fit_intercept': False, 'tol': 1.5950695115168866e-06, 'max_iter': 3825, 'weight_class_0': 1.0920750377564916, 'weight_class_1': 4.782653900286252, 'weight_class_2': 2.9480644522750326}

In [9]:
lg_params = {k: v for k, v in best_params.items() if k not in ['weight_class_0', 'weight_class_1', 'weight_class_2']}

lg = LogisticRegression(
    solver="saga",
    random_state=42,
    **lg_params
).fit(X_train.loc[:, ['xgb_0', 'xgb_1', 'xgb_2']], y_train.health_condition)

test_proba = lg.predict_proba(X_test.loc[:, ['xgb_0', 'xgb_1', 'xgb_2']])

weights = np.array([best_params['weight_class_0'], best_params['weight_class_1'], best_params['weight_class_2']])
weighted_probas = test_proba * weights

pred = np.argmax(weighted_probas, axis=1)

/home/junior/Documentos/GitHub/kaggle-competition-predicting-student-health/.venv/lib/python3.13/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


In [10]:
sub_labels = label_encoder.inverse_transform(pred)

# Submission

In [11]:
submission = pd.read_csv('../data/submission/sample_submission.csv')
submission['health_condition'] = sub_labels

submission.to_csv('../data/submission/submission_from_5.0.1_xgboost_submission.csv', index=False)

In [12]:
submission.head()

,id,health_condition
0,690088,unhealthy
1,690089,unhealthy
2,690090,at-risk
3,690091,at-risk
4,690092,unhealthy
